# Genotype–Phenotype Heterogeneity: Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR^2 clinical dataset using the `mlcroissant` library.

### Dataset Source
The dataset is structured following a Croissant schema and is available from:

`https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"\nName: {metadata.name}\nDescription: {metadata.description}\nIdentifier: {metadata.identifier}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

The dataset includes multiple clinical tables, each represented as a `RecordSet`. Below, we enumerate all available record sets and their respective fields using their `@id` values.

In [ ]:
# Explore available record sets and their fields (columns)
record_sets = [rs for rs in getattr(metadata, 'recordSets', [])]
if not record_sets:
    # Try alternative property name per Croissant spec
    record_sets = getattr(metadata, 'recordSet', [])

if record_sets:
    all_recordset_ids = []
    for rs in record_sets:
        print(f"RecordSet Name: {getattr(rs, 'name', 'N/A')}  |  @id: {rs['@id'] if '@id' in rs else getattr(rs, '@id', 'N/A')}")
        all_recordset_ids.append(rs['@id'] if '@id' in rs else getattr(rs, '@id', 'N/A'))
        # List the fields/columns for each record set
        cols = getattr(rs, 'fields', getattr(rs, 'columns', []))
        col_ids = []
        for col in cols:
            print(f"  Field: {getattr(col, 'name', 'N/A')}  |  @id: {col['@id'] if '@id' in col else getattr(col, '@id', 'N/A')}")
            col_ids.append(col['@id'] if '@id' in col else getattr(col, '@id', 'N/A'))
        print()
else:
    print("No record sets found in metadata.")

## 3. Data Extraction
Load data from each record set into Pandas DataFrames for analysis.

We use the `@id`s from above to reference each entity precisely.

In [ ]:
# If record sets found, load their data
dataframes = {}
if record_sets:
    for record_set in all_recordset_ids:
        print(f"Loading records for RecordSet: {record_set}")
        records = list(dataset.records(record_set=record_set))
        df = pd.DataFrame(records)
        dataframes[record_set] = df
        print(f"Columns: {df.columns.tolist()}")
        print(df.head(), "\n")
else:
    print("No record sets defined for extraction.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data by attributes.

Below, we select a numeric field (e.g., age at diagnosis), filter records, normalize values, and group by relevant clinical features. All fields are referenced by their `@id`.

In [ ]:
# Example: EDA on 'age at diagnosis' in the main clinical table
# Replace these with actual @id values from section 2 above
main_recordset_id = all_recordset_ids[0] if all_recordset_ids else None
df = dataframes[main_recordset_id] if main_recordset_id else None

# Placeholder for @id of 'age at diagnosis' column
numeric_field_id = '<insert age_at_diagnosis @id>'  # e.g., 'cr:ageAtDiagnosis'
# Placeholder for @id of grouping field (e.g. 'hirsutism' or 'phenotype')
group_field_id = '<insert group_field @id>'  # e.g., 'cr:hirsutism'

if df is not None and numeric_field_id in df.columns:
    threshold = 10
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"Normalized '{numeric_field_id}' for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by chosen field
    if group_field_id in df.columns:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"Grouped data by '{group_field_id}':")
        print(grouped_df.head())
else:
    print("Please update the 'numeric_field_id' and 'group_field_id' variables above with the proper @id values from the overview.")

## 5. Visualization
Visualize key numeric field distributions and relationships using matplotlib/seaborn.

All columns referenced by their `@id`.

In [ ]:
# Only visualize if DataFrame and numeric field are available
if df is not None and numeric_field_id in df.columns:
    plt.figure(figsize=(6, 4))
    sns.histplot(df[numeric_field_id].dropna(), bins=8, kde=True)
    plt.title(f"Distribution of '{numeric_field_id}' (Age at Diagnosis)")
    plt.xlabel("Age at Diagnosis")
    plt.ylabel("Count")
    plt.show()

    # If group field present, create a boxplot
    if group_field_id in df.columns:
        plt.figure(figsize=(6, 4))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.title(f"'{numeric_field_id}' by '{group_field_id}'")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()
else:
    print("Update the variables to include proper field @id values for visualization.")

## 6. Conclusion
This notebook demonstrates how to load, overview, and analyze a Croissant-structured clinical dataset with `mlcroissant`.

- We referenced all entities (record sets, fields, columns) using their unique `@id`.
- Data extraction and processing steps were done for each record set.
- Exploratory operations and visualizations were conducted on numeric fields, grouped by key clinical descriptors.
- For additional analysis, refer to the Croissant schema documentation and the `mlcroissant` library.